# Notebook 04 — Módulo M5: Ranking Híbrido (RRF)

Combina os rankings BM25 e KNN via **Reciprocal Rank Fusion** (Cormack et al., 2009).

```
RRF_score(d) = Σᵢ  1 / (k + rankᵢ(d))    k=60
```

Gera `runs/hybrid_rrf.trec` e compara métricas com os baselines.

In [ ]:
import sys
sys.path.insert(0, '..')
from src.retrievers import (build_bm25_index, search_bm25,
                             build_tfidf_index, search_knn,
                             reciprocal_rank_fusion)
from src.utils import load_corpus, load_queries, write_trec_run
from pathlib import Path

CORPUS_PATH  = '../data/corpus.jsonl'
QUERIES_PATH = '../eval/queries.tsv'
OUTPUT_PATH  = 'runs/hybrid_rrf.trec'
K_RETRIEVE   = 100   # documentos recuperados por sistema
K_RRF        = 60    # constante RRF (padrão literatura)

corpus  = load_corpus(CORPUS_PATH)
queries = load_queries(QUERIES_PATH)
print(f'Corpus: {len(corpus)} documentos | Queries: {len(queries)}')

In [ ]:
print('Construindo índices...')
bm25_index              = build_bm25_index(corpus)
vectorizer, tfidf_matrix = build_tfidf_index(corpus)
print('Índices prontos.')

In [ ]:
Path('runs').mkdir(exist_ok=True)
open(OUTPUT_PATH, 'w').close()

for qid, qtext in queries.items():
    run_bm25 = search_bm25(qtext, bm25_index, corpus, k=K_RETRIEVE)
    run_knn  = search_knn(qtext, vectorizer, tfidf_matrix, corpus, k=K_RETRIEVE)
    run_rrf  = reciprocal_rank_fusion([run_bm25, run_knn], k=K_RRF)
    write_trec_run(run_rrf, qid, 'hybrid_rrf', OUTPUT_PATH)
    print(f'{qid}: top-1 BM25={run_bm25[0][0]} | KNN={run_knn[0][0]} | RRF={run_rrf[0][0]}')

print(f'\nSalvo em {OUTPUT_PATH}')

## Análise qualitativa — 2 queries

Comparar o ranking de cada sistema lado a lado para entender quando o híbrido ajuda.

In [ ]:
id2doc = {d['arxiv_id']: d for d in corpus}

def show_rankings(qtext, top_n=5):
    run_bm25 = search_bm25(qtext, bm25_index, corpus, k=top_n)
    run_knn  = search_knn(qtext, vectorizer, tfidf_matrix, corpus, k=top_n)
    run_rrf  = reciprocal_rank_fusion([run_bm25, run_knn])[:top_n]
    print(f'Query: "{qtext}"\n')
    print(f'{'#':3} | {'BM25':40} | {'KNN':40} | {'RRF Híbrido':40}')
    print('-' * 130)
    for i in range(top_n):
        b = id2doc.get(run_bm25[i][0], {}).get('title', '')[:38]
        k = id2doc.get(run_knn[i][0], {}).get('title', '')[:38]
        r = id2doc.get(run_rrf[i][0], {}).get('title', '')[:38]
        print(f'{i+1:3} | {b:40} | {k:40} | {r:40}')
    print()

# Query 1: híbrido sparse+dense
show_rankings('hybrid retrieval sparse dense ranking fusion')

# Query 2: extração de cláusulas contratuais
show_rankings('contract clause extraction deep learning')